[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/electrokinetics/electrokinetics.ipynb)

# Electrokinetic Transport

This example computes the fluid flow and electric current that a pressure drop or an electric field drives through triangulated channels whose width and surface charge vary along them. A charged wall draws oppositely charged ions into a screening cloud beside it, which is called the electric double layer. Pushing the liquid with a pressure drop convects that cloud as an electric current, which is called the streaming current; applying an electric field pulls the cloud so that it drags the liquid along, which is called electro-osmosis, while individual ions also drift through the liquid by conduction. Onsager reciprocity equates those two cross responses even when channel geometry varies along the flow. Where channel width and wall charge vary periodically with zero net charge, local fields and double layers still pump fluid or stir closed recirculation cells, while straight-slit responses determine maximum mechanical-to-electrical power conversion.

Planar projective spacetime places channel vertices at unit projective weight along `origin`, joined with a degenerate quantity direction `v` that records transported fluid volume without equating volume to length. The term ***parcel*** here means a material element carrying extensive contents without projective weight, so a unit volume of solution is modelled as `volume` plus ionic charge along `time`, whereas an ion is modelled as pure charge without volume. Thermodynamic drives wedge minus each spatial potential gradient with its conjugate direction, `-(gradient ^ (pressure * v + potential * t))`, exerting body force `drive.commutator(parcel)` on parcels and spending power `drive & flux` on fluxes. Vertex stream values advance volume across triangle sides without accumulation, while the induced electric potential enforces zero net current divergence against every corner's simplex gradient, so `battery & flux(pump) == pump & flux(battery)` emerges directly from the symmetric viscous and conductive operators. [The capillary electrophoresis notebook](electrophoresis.ipynb) tracks neutral and charged dye bands carried and separated by these flows.

In vector-calculus notation, as in the lubrication analysis of [Ghosal (2004)](https://ghosal.mech.northwestern.edu/electrophoresis_review_ghosal.pdf) and the straight-slit power conversion of [van der Heyden, Bonthuis, Stein, Meyer and Dekker (2006)](https://ceesdekkerlab.nl/wp-content/uploads/2006_Heyden.pdf), fluid momentum and continuity split pressure and electrostatic body forces across $\eta\nabla^2\mathbf u = \nabla p + \rho\nabla\phi$ and $\nabla\cdot\mathbf u = 0$, or in stream-function coordinates $\mathbf u = (\partial_y\psi, -\partial_x\psi)$ across the fourth-order vorticity equation $\eta\nabla^4\psi = \partial_x\rho\,\partial_y\phi - \partial_y\rho\,\partial_x\phi$, coupled to charge conservation $\nabla\cdot(\sigma\nabla\phi - \rho\mathbf u) = 0$ and the Poisson–Boltzmann equation $\epsilon\nabla^2\phi = -\rho$, with cross-sectional volume and current fluxes assembled into a separate $2\times 2$ transport matrix $L_{ij}$ multiplying $(-\partial_x p, -\partial_x\phi)$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology
from examples.util.animation import show_animation
from examples.fluids.electrokinetics import render, scenarios
from examples.fluids.electrokinetics.numerical import coupled_solution

# Planar projective spacetime for channel geometry and electrodynamics, joined with volume as a transported quantity.
plane = Algebra("x+y+")
spacetime = Algebra("x+y+t-w0")
quantities = Algebra("v0")
ga = spacetime * quantities
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Antiscalar = ga.gatype.antiscalar()                                           # Antiscalar-
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Upright = Plane.derive.within(plane)                                          # Vector+
Drive = ga.gatype.bivector().derive.touching(plane, Algebra("t-v0"))          # Bivector+
Rotation = ga.gatype.bivector().derive.within(plane)                          # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
origin = mv.w.dual()                                                          # [] Point-
axis = mv.x.dual()                                                            # [] Point-
across = mv.y.dual()                                                          # [] Point-
time = -mv.t.dual()                                                           # [] Point-
volume = mv.v.dual()                                                          # [] Point-
unread = origin & time & volume                                               # [] Antitrivector-
pump, battery = mv.xv, mv.xt                                                  # [] Drive+, Drive+

## Periodic wavy channels

Sweeping a transverse chain of vertices around a longitudinal loop forms a closed triangulated strip with upper and lower walls and periodic ends. Channel width oscillates sinusoidally along the strip while each channel carries a zero-mean sinusoidal surface-charge wave of the same wavelength at a distinct phase lag. Placing vertices at unit projective weight along `origin` makes `strip.d[0] * strip_vertices` cancel `origin` on every edge, leaving pure spatial displacements supplemented by one period shift across the seam. Lengths are measured in bulk Debye lengths and wall charge densities in permittivity times thermal volts per Debye length, roughly 2 mC/m² in millimolar salt.

In [ ]:
strip, strip_places, edge_shifts, lower_wall, upper_wall, wall_charge_values, wall_charges = scenarios.channels(
    scenarios.n_stations, scenarios.n_across)
# Projective vertex placement at origin cancels in d[0], leaving spatial edge displacements across the strip.
strip_vertices = (origin + axis * strip_places[..., 0] + across * strip_places[..., 1]).field()  # Point-[V]
strip_edges = strip.d[0] * strip_vertices + (axis * edge_shifts).field()                         # Point-[E]
render.channels_figure(strip, strip_vertices, scenarios.n_across, scenarios.wavelength, wall_charges);

Wall strokes show positive surface charge in magenta and negative surface charge in green around the pale liquid interior of each periodic strip. From top to bottom, the surface-charge wave runs in phase with the channel width, a quarter wavelength behind it, and half a wavelength behind it.

## Equilibrium double layer

Thermal agitation and electrostatic energy settle each ion species into its Boltzmann concentration around the equilibrium electric potential, while Gauss's law equates dielectric flux leaving each dual cell to the ionic and wall charge inside it. The term ***simplex gradient*** here means the spatial plane for a triangle corner that vanishes along the facing side and rises by one to that corner, `(unread & sides) / double_areas`, built from incidence and exterior products before any metric enters. Material permittivity enters when unit corner potentials form electric drives `-(gradients ^ t)` and vacuum excitations `corner_drives.commutator(time) & time`, whose regressive pairing assembles the vertex outflow operator solved by damped Newton iteration.

In finite-element notation, the dielectric outflow operator is the scalar stiffness matrix $K_{ij} = \int \nabla\varphi_i\cdot\nabla\varphi_j\,\mathrm{d}A$ of piecewise-linear basis functions $\varphi_i$.

In [ ]:
# Aqueous potassium chloride valences and bulk concentrations for unit Debye length.
valences = np.array([1, -1])
bulk = np.array([0.5, 0.5])
n_newton_steps = 20
n_channels = len(scenarios.phases)

In [ ]:
def triangle_geometry(topology: Topology, edges: Point) -> tuple[Scalar, Point, Upright]:
    """Oriented `edges` (`[...] Point-[E]`) determine each triangle's area (`[...] Scalar+[F]`), facing sides (`[..., 3] Point-[F]`), and spatial simplex gradients (`[..., 3] Upright+[F]`)."""
    sides = topology.boundary[2, 1] * edges[..., None]                        # [..., 3] Point-[F]
    double_areas = sides[..., 0] & sides[..., 1] & unread                     # [...] Antiscalar-[F]
    gradients = (unread & sides) / double_areas[..., None]                    # [..., 3] Upright+[F]
    return double_areas.norm() / 2, sides, gradients


def vertex_form(topology: Topology, corner_form: Scalar) -> SparseExtensor:
    """Pullback by triangle-corner incidence assembles a local `[..., 3, 3]` form (`Scalar+[F]`) into a vertex operator (`[...] Scalar+[V <- V]`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    return (corners[:, None].adjugate() * spdiag(corner_form) * corners[None, :]).sum(axis=-1).sum(axis=-1)

In [ ]:
def double_layer(topology: Topology, edges: Point, wall_charge_field: Scalar) -> Scalar:
    """Damped Newton iteration solves the Poisson-Boltzmann balance on `topology` for the equilibrium potential (`[...] Scalar+[V]`)."""
    areas, _, gradients = triangle_geometry(topology, edges)
    corner_drives = -(gradients ^ mv.t)                                       # [..., 3] Drive+[F]
    excitations = corner_drives.commutator(time) & time                       # [..., 3] Flux-[F]
    outflow = vertex_form(topology, (corner_drives[..., :, None] & excitations[..., None, :]) * areas[..., None, None])
    dual_areas = topology.incidence[2, 0].sum(axis=0).adjugate() * (areas / 3)  # [...] Scalar+[V]
    potential = 0 * wall_charge_field                                         # [...] Scalar+[V]
    for _ in range(n_newton_steps):
        concentrations = bulk * (-valences * potential[..., None]).exp()      # [..., 2] Scalar+[V]
        residual = outflow * potential - dual_areas * (valences * concentrations).sum(axis=-1) - wall_charge_field
        susceptibility = (valences ** 2 * concentrations).sum(axis=-1)        # [...] Scalar+[V]
        step = ((outflow + dual_areas * susceptibility) * Scalar).solve(residual)  # [...] Scalar+[V]
        potential = potential - step.tanh()
    return potential


strip_potentials = double_layer(strip, strip_edges, mv.scalar(wall_charge_values[..., None]).field())  # [n_channels] Scalar+[V]
densities = (valences * bulk * (-valences * strip_potentials[..., None]).exp()).sum(axis=-1)           # [n_channels] Scalar+[V]

Across each channel the screening cloud sits in mechanical equilibrium. Excess ion concentration above the bulk creates an osmotic pressure whose drive `-(gradient ^ (osmotic_pressure * v))` pushes parcel volume away from the wall, while the double-layer potential creates an electric drive `-(gradient ^ (potential * t))` that pulls parcel charge back toward the wall. Both transverse drives act through the same commutator `drive.commutator(parcel)` and cancel across the resolved double layer.

In [ ]:
strip_areas, strip_sides, strip_gradients = triangle_geometry(strip, strip_edges)
corners = strip.incidence[2, 0]                                               # [3] Scalar+[F <- V]
strip_dual_areas = corners.sum(axis=0).adjugate() * (strip_areas / 3)         # Scalar+[V]
strip_concentrations = corners.mean(axis=0) * (bulk * (-valences * strip_potentials[..., None]).exp())  # [n_channels, 2] Scalar+[F]
strip_parcels = volume + (time * valences * strip_concentrations).sum(axis=-1)  # [n_channels] Point-[F]
osmotic_pressure = (bulk * ((-valences * strip_potentials[..., None]).exp() - 1)).sum(axis=-1)          # [n_channels] Scalar+[V]
osmotic_drive = -(((corners * osmotic_pressure[..., None]) * strip_gradients).sum(axis=-1) ^ mv.v)      # [n_channels] Drive+[F]
electric_drive = -(((corners * strip_potentials[..., None]) * strip_gradients).sum(axis=-1) ^ mv.t)     # [n_channels] Drive+[F]
# Area-weighted triangle body forces lumped onto vertex dual cells.
osmotic_force = (corners.sum(axis=0).adjugate() * (osmotic_drive.commutator(strip_parcels) * (strip_areas / 3))) / strip_dual_areas   # [n_channels] Point-[V]
electric_force = (corners.sum(axis=0).adjugate() * (electric_drive.commutator(strip_parcels) * (strip_areas / 3))) / strip_dual_areas  # [n_channels] Point-[V]

## Coupled Stokes flow and ion current

Acting by commutator on a fluid parcel, an axial pressure drop pushes volume alone, `pump.commutator(volume) == axis`, while an axial electric field pushes charge alone, `battery.commutator(time) == axis`, leaving pure charge parcels unaffected by pressure gradients without separate constitutive rules. Vertex stream values advance fluid along facing triangle sides by `sides / 2` so volume is conserved automatically, while dual-cell vorticity measured through `circulation_plane` supplies viscous resistance. Coupling advected parcel fluxes with corner potential drives `-(gradients ^ t)` and ionic conduction `drive.commutator(time) & time` yields one block-symmetric saddle system for the stream function and induced potential.

In [ ]:
viscosity = 1.0
# Dimensionless aqueous mobilities of potassium and chloride ions.
mobilities = np.array([4.27, 4.43])

In [ ]:
circulation_plane = (stack((mv.x, mv.y)) * (stack((mv.x, mv.y)) & Point)).sum(axis=0)  # [] Plane+ <- Point-


def transport(
    topology: Topology,
    edges: Point,
    potential: Scalar,
    periods: np.ndarray,
    lower_indices: np.ndarray,
    upper_indices: np.ndarray,
    drives: Drive,
) -> tuple[Scalar, Scalar, Flux]:
    """Coupled Stokes vorticity balance and charge conservation under `drives` (`[...] Drive+`) determine the stream function and induced potential (`[...] Scalar+[V]`) and period-averaged flux (`[...] Flux-`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    areas, sides, gradients = triangle_geometry(topology, edges)
    dual_areas = corners.sum(axis=0).adjugate() * (areas / 3)                 # [...] Scalar+[V]
    concentrations = corners.mean(axis=0) * (bulk * (-valences * potential[..., None]).exp())  # [..., 2] Scalar+[F]
    parcels = volume + (time * valences * concentrations).sum(axis=-1)        # [...] Point-[F]
    conductivity = (mobilities * valences ** 2 * concentrations).sum(axis=-1) # [...] Scalar+[F]
    flows = sides / 2                                                         # [..., 3] Point-[F]
    corner_drives = -(gradients ^ mv.t)                                       # [..., 3] Drive+[F]
    corner_currents = corner_drives.commutator(time) & time                   # [..., 3] Flux-[F]
    corner_velocities = flows / areas[..., None]                              # [..., 3] Point-[F]
    circulation = vertex_form(
        topology,
        (circulation_plane(corner_velocities[..., :, None]) & corner_velocities[..., None, :]) * areas[..., None, None],
    )
    viscous = viscosity * circulation * spdiag(1 / dual_areas) * circulation  # [...] Scalar+[V <- V]
    advected_fluxes = flows & parcels[..., None]                              # [..., 3] Flux-[F]
    coupling = vertex_form(topology, corner_drives[..., None, :] & advected_fluxes[..., :, None])
    conduction = vertex_form(
        topology, (corner_drives[..., :, None] & corner_currents[..., None, :]) * (conductivity * areas)[..., None, None])
    flow_load = (corners.adjugate() * (drives[..., None] & advected_fluxes)).sum(axis=-1)       # [...] Scalar+[V]
    applied_current = conductivity * (drives.commutator(time) & time)         # [...] Flux-[F]
    current_load = (corners.adjugate() * ((corner_drives & applied_current[..., None]) * areas[..., None])).sum(axis=-1)
    stream, induced_potential = coupled_solution(
        viscous, coupling, conduction, flow_load, current_load, topology.interior[0], upper_indices, int(lower_indices[0]))
    velocities = ((corners * stream[..., None]) * sides).sum(axis=-1) / (2 * areas)             # [...] Point-[F]
    field = drives + ((corners * induced_potential[..., None]) * corner_drives).sum(axis=-1)    # [...] Drive+[F]
    flux = (velocities & parcels) + conductivity * (field.commutator(time) & time)              # [...] Flux-[F]
    return stream, induced_potential, (flux * areas).sites.sum() / periods

## Onsager reciprocity

Driving each wavy channel separately with `pump` and `battery` and pairing both drives against the resulting fluxes produces a two-by-two response table per channel. Diagonal entries measure dissipated pumping and electrical power, while off-diagonal entries measure streaming current under pressure drop and electro-osmotic volume flow under electric field. Because the single coupling block `corner_drives & advected_fluxes` mediates both electro-osmotic driving and convected current against symmetric viscous and conductive forms, both cross responses match automatically.

In [ ]:
experiments = stack((pump, battery))                                          # [2] Drive+
streams, induced_potentials, channel_fluxes = transport(
    strip, strip_edges, strip_potentials, np.array(scenarios.wavelength),
    lower_wall, upper_wall, experiments[:, None],
)
readings = experiments[:, None, None] & channel_fluxes[None]                  # [2, 2, n_channels] Scalar+

In [ ]:
np.set_printoptions(precision=4)
print("Channel response matrices [pump, battery] x [pump, battery]:")
print(np.moveaxis(readings.to_array(), -1, 0))

## Electro-osmotic flow and circulation cells

Under battery driving alone, corner stream values reconstruct triangle velocities across each wavy strip and lump them by triangle area onto vertices alongside the total electrostatic potential.

In [ ]:
stream = streams[1]                                                           # [n_channels] Scalar+[V]
velocities = ((corners * stream[..., None]) * strip_sides).sum(axis=-1) / (2 * strip_areas)  # [n_channels] Point-[F]
speeds = (corners.sum(axis=0).adjugate() * (velocities * (strip_areas / 3))) / strip_dual_areas  # [n_channels] Point-[V]
axial_field = mv.x & battery.commutator(time)                                 # [] Scalar+
drive_potentials = induced_potentials[1] - axial_field * (mv.x & strip_vertices)  # [n_channels] Scalar+[V]
falls = axial_field * scenarios.wavelength * np.ones(n_channels)              # [n_channels] Scalar+
render.flow_figure(
    strip, strip_vertices, scenarios.n_across, scenarios.wavelength,
    densities, speeds, drive_potentials, falls, wall_charges,
);

Magenta and green shading shows positive and negative counter-ion clouds screening the opposite wall charges, dark streaks trace liquid velocity, and grey equipotential contours crowd in the narrow throats where current squeezes through a smaller cross-section. When wall charge oscillates in phase with width, strong throat fields pull net flow along the electric field, whereas a half-wavelength lag reverses net pumping despite zero net wall charge. At a quarter-wavelength lag, opposing wall pulls nearly cancel in net throughput and stir closed recirculation eddies, leaving only a slight residual drift from the mobility difference between chloride and potassium ions.

## Dye stripes under electro-osmotic pumping

Advecting periodic neutral dye stripes on one shared clock makes net pumping and internal recirculation visible together, while Fickian diffusion `gradients & fick_flux(gradients)` gently smooths sharp concentration fronts.

In [ ]:
fick_flux = (stack((axis, across)) * (Plane & stack((axis, across)))).sum(axis=0)  # [] Point- <- Plane+


def stripe_transport(flow_velocities: Point, dye: Scalar) -> Iterator[Scalar]:
    """Yields Crank-Nicolson frames of neutral `dye` (`[n_channels] Scalar+[V]`) advected by `flow_velocities` (`[n_channels] Point-[F]`) and smoothed by Fickian diffusion."""
    corner_inflows = strip_areas[..., None] * (strip_gradients & flow_velocities[..., None])    # [n_channels, 3] Scalar+[F]
    advection = vertex_form(strip, corner_inflows[..., :, None] * np.full(3, 1 / 3))            # [n_channels] Scalar+[V <- V]
    overlap_form = vertex_form(strip, strip_areas[..., None, None] * (np.ones((3, 3)) + np.eye(3)) / 12)
    diffusion = vertex_form(
        strip, strip_areas[..., None, None] * (strip_gradients[:, None] & fick_flux(strip_gradients[None, :])))
    transport_rate = scenarios.pumped_diffusivity * diffusion - advection                       # [n_channels] Scalar+[V <- V]
    forward_step = overlap_form / scenarios.pumped_dt - transport_rate / 2
    backward_step = (overlap_form / scenarios.pumped_dt + transport_rate / 2) * Scalar
    yield dye
    for _ in range(scenarios.n_pumped_frames):
        for _ in range(scenarios.pumped_stride):
            dye = backward_step.solve(forward_step * dye)
        yield dye


stripes = ((1 + np.cos(2 * np.pi * strip_places[:, 0] / scenarios.stripe_spacing)) / 2) ** scenarios.stripe_sharpness
dye = np.ones(n_channels) * mv.scalar(stripes[:, None]).field()               # [n_channels] Scalar+[V]
first_speed = (pump & channel_fluxes[1, 0]) / (2 * scenarios.mean_half_width) # [] Scalar+
show_animation(
    render.pumping_animation(
        strip, strip_vertices, scenarios.n_across, scenarios.wavelength,
        densities, speeds, drive_potentials, falls, wall_charges, stripe_transport(velocities / first_speed, dye),
    ),
    "electrokinetics_pumping",
    scenarios.frame_duration_ms,
)

Blue dye stripes overlay the charge clouds, velocity streaks, and equipotential lines of all three channels. Stripes in the top channel shear into forward chevrons and advance half a wavelength downstream, stripes in the bottom channel travel backward against the applied field, and stripes in the middle channel wrap around closed eddies with almost zero net translation.

## Viscous and Joule entropy production

Electrical power `battery & flux` supplied over one wavelength equals the integrated rate of heat dissipation across dual cells. The term ***duality map*** here means a linear map from an extrinsic rate or gradient to its conjugate intrinsic moment or flux, so `viscous_moment = patch * (Rotation & patch)` maps the dual-cell turning rate `xy * vorticity` to its resisting moment and produces viscous heat `rotation & (viscosity * viscous_moment(rotation))`, while conductive ion drift under the total field produces Joule heat `field & current`.

In [ ]:
patch = axis & across                                                         # [] Flux-
viscous_moment = patch * (Rotation & patch)                                   # [] Flux- <- Rotation+
corner_velocities = strip_sides / (2 * strip_areas)                           # [3] Point-[F]
circulation = vertex_form(
    strip, (circulation_plane(corner_velocities[:, None]) & corner_velocities[None, :]) * strip_areas)
rotation = mv.xy * ((circulation * stream) / strip_dual_areas)                # [n_channels] Rotation+[V]
viscous_heat = rotation & (viscosity * viscous_moment(rotation))              # [n_channels] Scalar+[V]
corner_drives = -(strip_gradients ^ mv.t)                                     # [3] Drive+[F]
field = battery + ((corners * induced_potentials[1][..., None]) * corner_drives).sum(axis=-1)   # [n_channels] Drive+[F]
current = (mobilities * valences ** 2 * strip_concentrations).sum(axis=-1) * (field.commutator(time) & time)  # [n_channels] Flux-[F]
joule_heat = (corners.sum(axis=0).adjugate() * ((field & current) * strip_areas / 3)) / strip_dual_areas      # [n_channels] Scalar+[V]
heat = viscous_heat + joule_heat                                              # [n_channels] Scalar+[V]
render.heat_figure(
    strip, strip_vertices, scenarios.n_across, scenarios.wavelength,
    heat, drive_potentials, falls, wall_charges,
);

Orange shading maps local heat production from white at zero to brightest orange at the peak dissipation across all three channels. Ionic Joule heating accounts for nearly all dissipation and concentrates in the narrow throats where equipotentials crowd, peaking sharpest in the middle channel whose throat walls carry zero surface charge and therefore fewer mobile counter-ions.

## Electrokinetic power generation in straight slits

Pumping electrolyte through a straight charged slit connected across an external electrical load drives a streaming current against the induced voltage drop to deliver electrical power. Maximum conversion efficiency depends only on the dimensionless electrokinetic coupling `coupling` through `coupling / (1 + (1 - coupling).square_root()).squared()`. Solving the logarithmic grid of slit widths and wall charges takes about five seconds.

In [ ]:
n_widths, n_charges = scenarios.n_widths, scenarios.n_charges
generators, generator_places, generator_shifts, generator_lower, generator_upper, generator_charges, lengths = scenarios.generators()
generator_vertices = (origin + axis * generator_places[..., 0] + across * generator_places[..., 1]).field()  # [n_widths, n_charges] Point-[V]
generator_edges = generators.d[0] * generator_vertices + (axis * generator_shifts).field()                   # [n_widths, n_charges] Point-[E]
generator_potentials = double_layer(generators, generator_edges, mv.scalar(generator_charges[..., None]).field())
_, _, generator_fluxes = transport(
    generators, generator_edges, generator_potentials, lengths,
    generator_lower, generator_upper, experiments[:, None, None],
)                                                                             # [2, n_widths, n_charges] Flux-
generator_readings = experiments[:, None, None, None] & generator_fluxes[None]  # [2, 2, n_widths, n_charges] Scalar+
coupling = generator_readings[0, 1].squared() / (generator_readings[0, 0] * generator_readings[1, 1])  # [n_widths, n_charges] Scalar+
efficiency = coupling / (1 + (1 - coupling).square_root()).squared()          # [n_widths, n_charges] Scalar+
render.efficiency_figure(scenarios.half_widths, scenarios.surface_charges, efficiency);

Blue shading and percent contours show maximum conversion efficiency over slit half-width and wall surface-charge magnitude on logarithmic axes. In slits narrower than a Debye length, efficiency depends on width and wall charge through their product and peaks near 6.4 percent along the ridge where half-width equals roughly eight Gouy–Chapman lengths, `2 / abs(surface_charge)`. Weaker wall charges leave too few counter-ions to convect, stronger wall charges pin counter-ions against the no-slip walls where conduction outpaces advection, and slits wider than a Debye length admit bulk co-ions that short-circuit the streaming current.

In [ ]:
# checks
# Pressure drop pushes volume alone and battery pushes charge alone.
drive_forces = stack((
    pump.commutator(volume), pump.commutator(time),
    battery.commutator(time), battery.commutator(volume),
))
assert (((mv.x & drive_forces) - np.array([1, 0, 1, 0])).abs() < 1e-10).all()
# Transverse osmotic and electrostatic body forces cancel across the double layer up to triangle resolution.
assert (mv.y & (osmotic_force + electric_force)).abs().max() < 0.1 * (mv.y & osmotic_force).abs().max()
# Onsager response matrices are symmetric and strictly dissipative.
assert ((readings[0, 1] - readings[1, 0]).abs() < 1e-10 * readings[0, 0]).all()
assert (readings[0, 0] > 0).all() and (readings[0, 0] * readings[1, 1] > readings[0, 1].squared()).all()
# Wavy channels pump forward in phase, backward at half-wave lag, and nearly zero at quarter-wave lag.
pumped_rates = pump & channel_fluxes[1]
assert pumped_rates[0] > 0 and pumped_rates[2] < 0 and pumped_rates[1].abs() < 0.05 * pumped_rates[0]
# Total viscous plus Joule heating equals electrical power supplied by the battery.
heat_power = (heat * strip_dual_areas).sites.sum()                            # [n_channels] Scalar+
electric_power = (battery & channel_fluxes[1]) * scenarios.wavelength         # [n_channels] Scalar+
assert ((heat_power - electric_power).abs() < 1e-10 * electric_power).all()
# Straight-slit generator efficiency peaks near 6.4 percent around eight Gouy-Chapman lengths.
best = np.unravel_index(efficiency.argmax(), efficiency.shape)
assert (efficiency[best] - 0.064).abs() < 3e-3
assert 6 < scenarios.half_widths[best[0]] * np.abs(scenarios.surface_charges[best[1]]) / 2 < 11